Creates per-species 100 kb  and percent-based genomic windows and counts intact, partial, and solo elements per window per family. Computes solo ratios and mean LTR identity per window. Produces {species}_percent_counts.tsv and {species}_counts.tsv in data/ratio_tables/.
Input: Master element tables, chromosome sizes, metadata
Output: data/ratio_tables/{species}_percent_counts.tsv — the "bin data" used by DTW clustering, epigentic heatmap, solo_plots, etc.

In [2]:
import pandas as pd
import pyranges as pr

Genome indices prepared with:

```bash
module add samtools

samples=(
  Poa_supina
  Poa_infirma
  Poa_chaixii
  Poa_annua
  Hordeum_vulgare
  Secale_cereale
  Triticum_aestivum
  Triticum_monococcum
  Brachypodium_hybridum
  Brachypodium_distachyon
  Brachypodium_stacei
  Oryza_sativa
  Oryza_rufipogon
  Oryza_glaberrima
  Oryza_barthii
  Oryza_grandiglumis
  Oryza_punctata
  Oryza_australiensis
  Oryza_coarctata
  Oryza_ridleyi
  Oryza_longiglumis
  Oryza_brachyantha
  Oryza_meyeriana
  Zoysia_sinica
  Zoysia_japonica
  Panicum_miliaceum
  Juncus_squarrosus
  Juncus_effusus
  Luzula_pallescens
  Luzula_sylvatica
  Cyperus_fuscus
  Cyperus_iria
  Schoenoplectus_tabernaemontani
  Schoenoplectus_triqueter
  Schoenoplectus_lacustris
  Bolboschoenus_planiculmis
  Eleocharis_quinqueflora
  Eleocharis_dulcis
  Carex_riparia
  Carex_rostrata
  Carex_hirta
  Carex_nigra
  Carex_elata
  Carex_littledalei
  Carex_depauperata
  Carex_sylvatica
  Carex_acutiformis
  Carex_caryophyllea
  Carex_pendula
  Carex_extensa
  Carex_distans
  Carex_divulsa
  Carex_arenaria
  Carex_echinata
  Carex_myosuroides
  Carex_laevigata
  Scirpus_sylvaticus
  Eriophorum_angustifolium
  Eriophorum_vaginatum
  Trichophorum_cespitosum
  Rhynchospora_pubera
  Rhynchospora_breviuscula
  Rhynchospora_tenuis
  Chionographis_japonica
  Cuscuta_campestris
  Cuscuta_epithymum
  Chamaelirium_luteum

)

for genome in "${samples[@]}"; do
  src="data/genomes/${genome}.fna.gz"
  dst="data/genomes/${genome}.fna.bgz"
  fai="${dst}.fai"
  gzi="${dst}.gzi"

  echo "Processing ${genome}"

  if [ ! -f "$src" ]; then
    echo "  Skipping: source not found: $src" >&2
    continue
  fi

  # Create BGZF-compressed FASTA if missing
  if [ ! -f "$dst" ]; then
    echo "  Creating BGZF: $dst"
    gzip -dc "$src" | bgzip > "$dst"
  else
    echo "  BGZF exists: $dst"
  fi

  # Create indexes if missing (.fai and .gzi)
  if [ ! -f "$fai" ] || [ ! -f "$gzi" ]; then
    echo "  Indexing: $dst"
    samtools faidx "$dst"
  else
    echo "  Indexes exist: $fai and $gzi"
  fi
done
```

In [ ]:
sample_order_path = "data/metadata.tsv"
sample_order = pd.read_csv(sample_order_path, sep='\t')['Species'].tolist()

In [ ]:
def load_elements(intact_path, disrupted_path, solo_path):
    intact_all = pd.read_csv(intact_path, sep='\t')
    disrupted_all = pd.read_csv(disrupted_path, sep='\t')
    solo_all = pd.read_csv(solo_path, sep='\t')

    return intact_all, disrupted_all, solo_all


def filter_elements(intact_all, disrupted_all, solo_all, sample):
    intact = intact_all[intact_all['sample'] == sample]
    disrupted = disrupted_all[disrupted_all['sample'] == sample]
    solo = solo_all[solo_all['sample'] == sample]

    intact_bed = intact[['chr', 'start', 'end', 'Class', 'LTR_identity']]
    disrupted_bed = disrupted[['chr', 'start', 'end', 'Class']]
    solo_bed = solo[['chr', 'start', 'end', 'Class']]

    return intact_bed, disrupted_bed, solo_bed

def make_windows(chromosomes, index_path, window_size=1_000_000, window_step=100_000):
    column_names = ['chr', 'length', 'byte_index', 'bases_per_line', 'bytes_per_line']
    index = pd.read_csv(index_path, sep='\t', header=None, names=column_names)
    windows = pd.DataFrame(columns=['chr', 'bin_ID','bin_start', 'bin_end'])
    for chromosome in chromosomes:
        if chromosome not in index['chr'].values:
            raise ValueError(f"Chromosome {chromosome} not found in index file.")
        chr_length = index.loc[index['chr'] == chromosome, 'length'].values[0]
        chr_windows = pd.DataFrame(columns=['chr', 'bin_ID','bin_start', 'bin_end'])
        bin_ID = 1
        bin_start = 1
        bin_end = window_size
        while bin_end <= chr_length:
            chr_windows = pd.concat([chr_windows, pd.DataFrame({'chr': [chromosome], 'bin_ID': [bin_ID], 'bin_start': [bin_start], 'bin_end': [bin_end]})], ignore_index=True)
            bin_ID += 1
            bin_start += window_step
            bin_end += window_step
        if bin_start < chr_length:
            chr_windows = pd.concat([chr_windows, pd.DataFrame({'chr': [chromosome], 'bin_ID': [bin_ID], 'bin_start': [bin_start], 'bin_end': [chr_length]})], ignore_index=True)
        windows = pd.concat([windows, chr_windows], ignore_index=True)
    windows['bin_ID'] = windows['bin_ID'].astype(int)
    windows['bin_start'] = windows['bin_start'].astype(int)
    windows['bin_end'] = windows['bin_end'].astype(int)
    return windows

def make_windows_percent(chromosomes, index_path, window_percent_size=5, window_percent_step=1):
    column_names = ['chr', 'length', 'byte_index', 'bases_per_line', 'bytes_per_line']
    index = pd.read_csv(index_path, sep='\t', header=None, names=column_names)
    windows = pd.DataFrame(columns=['chr', 'bin_ID','bin_start', 'bin_end'])
    for chromosome in chromosomes:
        if chromosome not in index['chr'].values:
            raise ValueError(f"Chromosome {chromosome} not found in index file.")
        chr_length = index.loc[index['chr'] == chromosome, 'length'].values[0]
        chr_windows = pd.DataFrame(columns=['chr', 'bin_ID','bin_start', 'bin_end'])
        window_size = int(chr_length * window_percent_size / 100)
        window_step = int(chr_length * window_percent_step / 100)
        bin_ID = 1
        bin_start = 1
        bin_end = window_size
        while bin_end <= chr_length:
            chr_windows = pd.concat([chr_windows, pd.DataFrame({'chr': [chromosome], 'bin_ID': [bin_ID], 'bin_start': [bin_start], 'bin_end': [bin_end]})], ignore_index=True)
            bin_ID += 1
            bin_start += window_step
            bin_end += window_step
        if bin_start < chr_length:
            chr_windows = pd.concat([chr_windows, pd.DataFrame({'chr': [chromosome], 'bin_ID': [bin_ID], 'bin_start': [bin_start], 'bin_end': [chr_length]})], ignore_index=True)
        windows = pd.concat([windows, chr_windows], ignore_index=True)
    windows['bin_ID'] = windows['bin_ID'].astype(int)
    windows['bin_start'] = windows['bin_start'].astype(int)
    windows['bin_end'] = windows['bin_end'].astype(int)
    return windows

def counts_in_windows_old(intact, disrupted, solo, windows, te_families):
    # Convert all DataFrames to PyRanges
    pr_windows = pr.PyRanges(windows.rename(columns={"chr": "Chromosome", "bin_start": "Start", "bin_end": "End"}))
    pr_intact = pr.PyRanges(intact.rename(columns={"chr": "Chromosome", "start": "Start", "end": "End"}))
    pr_disrupted = pr.PyRanges(disrupted.rename(columns={"chr": "Chromosome", "start": "Start", "end": "End"}))
    pr_solo = pr.PyRanges(solo.rename(columns={"chr": "Chromosome", "start": "Start", "end": "End"}))

    # Count total overlaps
    windows["all_intact_count"] = pr_windows.join(pr_intact, apply_strand_suffix=False).df.groupby(["Chromosome", "Start", "End"], observed=True).size().reindex(pr_windows.df.set_index(["Chromosome", "Start", "End"]).index, fill_value=0).values
    windows["all_disrupted_count"] = pr_windows.join(pr_disrupted, apply_strand_suffix=False).df.groupby(["Chromosome", "Start", "End"], observed=True).size().reindex(pr_windows.df.set_index(["Chromosome", "Start", "End"]).index, fill_value=0).values
    windows["all_solo_count"] = pr_windows.join(pr_solo, apply_strand_suffix=False).df.groupby(["Chromosome", "Start", "End"], observed=True).size().reindex(pr_windows.df.set_index(["Chromosome", "Start", "End"]).index, fill_value=0).values

    for family in te_families:
        label = family.split('|')[-1]

        # Filter by family first, then convert
        fam_intact = pr.PyRanges(intact[intact['Class'] == family].rename(columns={"chr": "Chromosome", "start": "Start", "end": "End"}))
        fam_disrupted = pr.PyRanges(disrupted[disrupted['Class'] == family].rename(columns={"chr": "Chromosome", "start": "Start", "end": "End"}))
        fam_solo = pr.PyRanges(solo[solo['Class'] == family].rename(columns={"chr": "Chromosome", "start": "Start", "end": "End"}))

        index = pr_windows.df.set_index(["Chromosome", "Start", "End"]).index

        joined = pr_windows.join(fam_intact, apply_strand_suffix=False).df
        if not joined.empty:
            counts = joined.groupby(["Chromosome", "Start", "End"], observed=True).size()
        else:
            counts = pd.Series(dtype=int)
        windows[f"{label}_intact_count"] = counts.reindex(index, fill_value=0).values

        joined = pr_windows.join(fam_disrupted, apply_strand_suffix=False).df
        if not joined.empty:
            counts = joined.groupby(["Chromosome", "Start", "End"], observed=True).size()
        else:
            counts = pd.Series(dtype=int)
        windows[f"{label}_disrupted_count"] = counts.reindex(index, fill_value=0).values

        joined = pr_windows.join(fam_solo, apply_strand_suffix=False).df
        if not joined.empty:
            counts = joined.groupby(["Chromosome", "Start", "End"], observed=True).size()
        else:
            counts = pd.Series(dtype=int)

        windows[f"{label}_solo_count"] = counts.reindex(index, fill_value=0).values
        
    # Convert all count columns to int  
    count_cols = windows.columns[windows.columns.str.contains('_count')]
    for col in count_cols:
        if col in windows.columns:
            windows[col] = windows[col].astype(int)

    return windows

def counts_in_windows(intact, disrupted, solo, windows, te_families):
    # Convert all DataFrames to PyRanges
    pr_windows = pr.PyRanges(windows.rename(columns={"chr": "Chromosome", "bin_start": "Start", "bin_end": "End"}))
    pr_intact = pr.PyRanges(intact.rename(columns={"chr": "Chromosome", "start": "Start", "end": "End"}))
    pr_disrupted = pr.PyRanges(disrupted.rename(columns={"chr": "Chromosome", "start": "Start", "end": "End"}))
    pr_solo = pr.PyRanges(solo.rename(columns={"chr": "Chromosome", "start": "Start", "end": "End"}))

    # Precompute index
    index = pr_windows.df.set_index(["Chromosome", "Start", "End"]).index

    # Count total overlaps
    joined = pr_windows.join(pr_intact, apply_strand_suffix=False).df
    windows["all_intact_count"] = (
        joined.groupby(["Chromosome", "Start", "End"], observed=True).size()
        .reindex(index, fill_value=0).values
    )
    if not joined.empty:
        mean_identity = joined.groupby(["Chromosome", "Start", "End"], observed=True)["LTR_identity"].mean()
        windows["all_intact_LTR_identity"] = mean_identity.reindex(index, fill_value=float("nan")).values
    else:
        windows["all_intact_LTR_identity"] = float("nan")

    joined = pr_windows.join(pr_disrupted, apply_strand_suffix=False).df
    windows["all_disrupted_count"] = (
        joined.groupby(["Chromosome", "Start", "End"], observed=True).size()
        .reindex(index, fill_value=0).values
    )

    joined = pr_windows.join(pr_solo, apply_strand_suffix=False).df
    windows["all_solo_count"] = (
        joined.groupby(["Chromosome", "Start", "End"], observed=True).size()
        .reindex(index, fill_value=0).values
    )

    # Family-specific counts and LTR identities
    for family in te_families:
        label = family.split("|")[-1]

        fam_intact = pr.PyRanges(
            intact[intact["Class"] == family].rename(
                columns={"chr": "Chromosome", "start": "Start", "end": "End"}
            )
        )
        fam_disrupted = pr.PyRanges(
            disrupted[disrupted["Class"] == family].rename(
                columns={"chr": "Chromosome", "start": "Start", "end": "End"}
            )
        )
        fam_solo = pr.PyRanges(
            solo[solo["Class"] == family].rename(
                columns={"chr": "Chromosome", "start": "Start", "end": "End"}
            )
        )

        joined = pr_windows.join(fam_intact, apply_strand_suffix=False).df
        if not joined.empty:
            counts = joined.groupby(["Chromosome", "Start", "End"], observed=True).size()
            means = joined.groupby(["Chromosome", "Start", "End"], observed=True)["LTR_identity"].mean()
        else:
            counts = pd.Series(dtype=int)
            means = pd.Series(dtype=float)

        windows[f"{label}_intact_count"] = counts.reindex(index, fill_value=0).values
        windows[f"{label}_LTR_identity"] = means.reindex(index, fill_value=float("nan")).values

        joined = pr_windows.join(fam_disrupted, apply_strand_suffix=False).df
        if not joined.empty:
            counts = joined.groupby(["Chromosome", "Start", "End"], observed=True).size()
        else:
            counts = pd.Series(dtype=int)
        windows[f"{label}_disrupted_count"] = counts.reindex(index, fill_value=0).values

        joined = pr_windows.join(fam_solo, apply_strand_suffix=False).df
        if not joined.empty:
            counts = joined.groupby(["Chromosome", "Start", "End"], observed=True).size()
        else:
            counts = pd.Series(dtype=int)
        windows[f"{label}_solo_count"] = counts.reindex(index, fill_value=0).values

    # Convert all count columns to int
    count_cols = windows.columns[windows.columns.str.contains("_count")]
    for col in count_cols:
        windows[col] = windows[col].astype(int)

    return windows


intact_path = "data/intact_elements_dante.tsv"
disrupted_path = "data/partial_elements_dante.tsv"
solo_path = "data/solo_elements.tsv"

window_size = 1_000_000
window_step = 100_000
window_percent_size = 5
window_percent_step = 1

intact_all, disrupted_all, solo_all = load_elements(intact_path, disrupted_path, solo_path)

for sample in sample_order:
    print(f"Processing sample: {sample}")
    if sample not in intact_all['sample'].values:
        print(f"Sample {sample} not found in intact elements. Skipping.")
        continue
    index_path = f"data/genomes/{sample}.fna.bgz.fai"
    intact, disrupted, solo = filter_elements(intact_all, disrupted_all, solo_all, sample)
    chromosomes = pd.concat([intact, disrupted, solo])['chr'].unique()
    te_families = pd.concat([intact, disrupted, solo])['Class'].unique()
    windows = make_windows(chromosomes, index_path, window_size, window_step)
    counts_df= counts_in_windows(intact, disrupted, solo, windows, te_families)
    counts_df.to_csv(f"data/ratio_tables/{sample}_counts.tsv", sep='\t', index=False)
    percent_windows = make_windows_percent(chromosomes, index_path, window_percent_size, window_percent_step)
    counts_percent_df = counts_in_windows(intact, disrupted, solo, percent_windows, te_families)
    counts_percent_df.to_csv(f"data/ratio_tables/{sample}_percent_counts.tsv", sep='\t', index=False)


